# 07 — Task Decomposition and Workflow Prompting

## Scenario
Northstar receives an email requesting a refund. Our policy states that refunds are only valid if the purchase was made within the last 30 days. We have a mock database function to check purchase dates.

**The Danger:** A naive approach tries to do everything in one prompt. This forces the LLM to hallucinate database state or policy compliance because it doesn't actually have access to the deterministic data.

## Step 1: The "Do Everything" Baseline (Anti-Pattern)

Watch what happens when we ask the LLM to handle the whole process without giving it a way to check the actual database.

## Step 2: The Sequential Workflow

We break the task down into a pipeline. 
1. **LLM Node:** Extract the Order ID.
2. **Deterministic Node:** Python checks the database.
3. **LLM Node:** Draft the response using the concrete DB fact.

## Conclusion

By decomposing the task, we prevented a hallucination, isolated the deterministic logic (the database check) from the fuzzy logic (reading/writing emails), and created observable trace points (we know exactly what Order ID was extracted).

In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab07 import *

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: The "Do Everything" Baseline (Anti-Pattern)

In [ ]:
request = next(r for r in build_requests() if r.case_id == "i07/naive/original-email")
print("SYSTEM:\n", request.system)
print("USER:\n", request.messages[0].text)
response = client.generate(request)
print("RECORDED RESPONSE:\n", response.text)
parsed = Draft.model_validate_json(response.text)
print("PARSED:", parsed)
assert check_constraints(parsed.answer, forbidden_phrases=("approved",))

## Step 2: The Sequential Workflow

In [ ]:
for case in CASES:
    trace = run_workflow(client, case["email"])
    print(case["id"], trace.steps, trace.terminal_state)
    assert trace.terminal_state == case["expected_terminal"]
assert run_workflow(client, CASES[2]["email"]).terminal_state == "clarification_required"

## Step 3: Deterministic policy gate

In [ ]:
for case in CASES[:2]:
    print(case["email"])
    print("eligibility:", refund_eligible(case["expected_order_id"]))
assert refund_eligible("ORD-8812") == "ineligible"
assert refund_eligible("ORD-8813") == "eligible"

## Takeaway

This replay-backed experiment makes the application control and measured trade-off explicit.

## References

See the course README for the references and further reading.